> 📌 このノートブックは [Learn-Prompt-Hacking](https://github.com/TrustAI-laboratory/Learn-Prompt-Hacking) の日本語版です。

# GenAI Application Framework Methodology（生成AIアプリケーションフレームワーク方法論）とは

生成 AI によるプロンプティング問題解決法は、生成 AI 分野における問題解決のフレームワークです。GenAI が適切なソリューションかどうかの判断、プロンプトエンジニアリングの適用方法、ツールの選定などに役立ちます。

ここでは 5 つのステップをそれぞれ解説し、その後この方法論を使ったケーススタディを紹介します。


# 5 つのステップ（SEPAL）

## 1、State your problem（問題を明確にする）

最初のステップは問題を明確にすることです。潜在的な解決策に飛びつかず、直面している問題を明確に言語化します。

例：「顧客が当社製品の機能について問い合わせを持っており、それに対応する必要がある。対応できないと潜在的なビジネスを逃してしまう。」


## 2、Examine relevant information（関連情報を調査する）

問題を明確にしたら、次は関連情報を調査します。

類似の問題とその解決策のリサーチ、問題のコンテキストの調査、問題に関連するデータの分析などが含まれます。

また、関連するプロンプトや GenAI ツール（LangChain、PromptAppGPT、Promptify、PromptFlow など）の調査も含まれます。

このステップは、問題のニュアンスを理解し、解決のための潜在的なアプローチを特定するために重要です。この時点で、GenAI が問題に適しているかどうかを判断できるはずです。


## 3、Propose a solution（ソリューションを提案する）

関連情報を調査したら、問題への対処法がより明確になっているはずです。ここでソリューションを提案します。

これはプロンプト、新しいツール、または既存ツールの新しい使い方かもしれません。ソリューションは、明確にした問題と調査した情報に直接結びついている必要があります。

## 4、Adjust the solution（ソリューションを調整する）

ソリューション（プロンプトやツールなど）を選んだら、次はフィードバックとテストに基づいて調整します。

ユーザーがプロンプトとどう対話するかのテストの設定、ユーザーからのフィードバック収集、自身の直感や専門知識に基づく調整などが含まれます。ここがプロンプトエンジニアリングの出番です！

## 5、Launch your solution（ソリューションを展開する）

最後のステップはソリューションの展開です。製品への統合、プラットフォームへの公開、またはユーザーとのやり取りで使い始めることなどが考えられます。

プロンプティング方法論はサイクルであり、直線的なプロセスではありません。ソリューション展開後も、パフォーマンスを継続的にモニタリングし、必要に応じて調整を行います。これらのステップは頭文字を取って **SEPAL** と覚えましょう！

# ケーススタディ：帽子情報ボットの作成

## セットアップ

LLM API を使うための初期設定を行います。事前に [00_Setup.ipynb](00_Setup.ipynb) を完了してください。

In [ ]:
%pip install -q openai

import sys; sys.path.insert(0, "..")
from utils.llm_client import call_llm, md_print, create_client

## チャットボットの作成

In [ ]:
# チャットボットクラスの作成

from utils.llm_client import DEFAULT_MODEL

class ChatBot:
    def __init__(self):
        # 会話履歴を保持するリスト
        self.context = []
        self.client = create_client()

    def new_message(self, prompt):
        # ユーザーのプロンプトをコンテキストに追加
        self.context.append({"role": "user", "content": prompt})

        # アシスタントの応答を生成
        completion = self.client.chat.completions.create(
          model=DEFAULT_MODEL,
          messages=self.context
        )

        # アシスタントの応答を解析
        chat_response = completion.choices[0].message.content

        # アシスタントの応答をコンテキストに追加
        self.context.append({"role": "assistant", "content": chat_response})

        # 会話を表示
        for message in self.context:
            if message["role"] == "user":
                md_print(f'User: {message["content"]}')
            else:
                md_print(f'LLM: {message["content"]}')

## 問題分析

Learn Prompting Method を使ってゼロからチャットボットを作成するケーススタディを見てみましょう。このケースでは、帽子に関するユーザーの質問コレクションがあります。

1. **State your problem（問題の明確化）**: 帽子の種類、歴史、着用方法について大量のユーザーからの問い合わせがある。対応できないと潜在的なビジネスを逃してしまう。

2. **Examine relevant information（関連情報の調査）**: 収集したユーザーの問い合わせを分析する。最も多い質問は、特定の帽子の歴史、正しい着用方法、手入れ方法であることがわかる。また、既存のチャットボットのコンテキスト長、料金、速度や、問題解決に役立つ GenAI ツールも調査する。

3. **Propose a solution（ソリューションの提案）**: 分析結果に基づき、これら 3 種類の質問に回答できる LLM チャットボットを作成することにする。初期プロンプトを作成する：

In [ ]:
# チャットボットのインスタンスを作成
chatbot1 = ChatBot()

# --- 元の英語プロンプト（参考用） ---
# guidance_prompt ="""
# You are a knowledgeable hat historian who has studied the history, styles, and
# proper ways to wear various types of hats. A user asks you a question about hats.
# Respond to their query in a helpful and informative manner: {USER_INPUT}
# """

# 初期プロンプト
guidance_prompt ="""
あなたは帽子の歴史、スタイル、正しいかぶり方を研究してきた知識豊富な帽子の歴史家です。
ユーザーが帽子について質問しています。
親切で有益な方法でクエリに応答してください: {USER_INPUT}
"""

# --- 元の英語プロンプト（参考用） ---
# # the {USER_INPUT} will be replaced by real user input.
# guidance_prompt.format(USER_INPUT="what is price of the hat?")

# {USER_INPUT} は実際のユーザー入力に置き換えられる
guidance_prompt = guidance_prompt.format(USER_INPUT="帽子の値段はいくらですか？")

chatbot1.new_message(guidance_prompt)

4. **Adjust the solution（ソリューションの調整）**: 初期プロンプトを少人数のユーザーグループでテストし、フィードバックを収集する。フィードバックに基づき、プロンプトをもっと親しみやすく、堅すぎないものにする必要があることがわかった。

プロンプトを調整する：

In [ ]:
# チャットボットのインスタンスを作成
chatbot1 = ChatBot()

# --- 元の英語プロンプト（参考用） ---
# guidance_prompt ="""
# You are a hat enthusiast with a wealth of knowledge about the history, styles,
# and etiquette of wearing various types of hats. A user is curious about hats and
# asks you a question. Respond to their query in a friendly and informative manner.
# """

# 調整後のプロンプト（よりフレンドリーに）
guidance_prompt ="""
あなたは帽子の歴史、スタイル、かぶり方のエチケットについて豊富な知識を持つ帽子愛好家です。
ユーザーが帽子について興味を持ち、質問しています。
フレンドリーかつ有益な方法で応答してください: {USER_INPUT}
"""

# --- 元の英語プロンプト（参考用） ---
# # the {USER_INPUT} will be replaced by real user input.
# guidance_prompt.format(USER_INPUT="what is price of the hat?")

# {USER_INPUT} は実際のユーザー入力に置き換えられる
guidance_prompt = guidance_prompt.format(USER_INPUT="帽子の値段はいくらですか？")

chatbot1.new_message(guidance_prompt)

さらにユーザーテストを行った結果、市場のセグメンテーションが必要であることがわかった：
* 帽子の歴史に興味がある人はフォーマルなアプローチを好む
* スタイルや着用方法に興味がある人はカジュアルなボットを好む

ユーザーの質問に基づいてどちらのタイプかを判定する、ルーティングプロンプトを開発する：


In [ ]:
# チャットボットのインスタンスを作成
chatbot1 = ChatBot()

# --- 元の英語プロンプト（参考用） ---
# intension_input ="""
# You are an AI that understands the nuances of hat-related queries.
# Based on the user's question, determine if they are more interested in the formal
# history of hats or the informal style and wearing of hats.
# Respond with 'Formal' for history-related queries and 'Informal' for style and
# wearing-related queries.
# """

# ルーティングプロンプト（フォーマル/カジュアルの判定）
intension_input ="""
あなたは帽子に関するクエリのニュアンスを理解する AI です。
ユーザーの質問に基づいて、帽子のフォーマルな歴史に興味があるのか、
カジュアルなスタイルやかぶり方に興味があるのかを判定してください。
歴史関連のクエリには「Formal」、スタイルやかぶり方関連のクエリには「Informal」と応答してください。
ユーザーからの質問: {USER_INPUT}
"""

# --- 元の英語プロンプト（参考用） ---
# # the {USER_INPUT} will be replaced by real user input.
# intension_input.format(USER_INPUT="what is price of the hat?")

# {USER_INPUT} は実際のユーザー入力に置き換えられる
intension_input = intension_input.format(USER_INPUT="帽子の値段はいくらですか？")

chatbot1.new_message(intension_input)

Langchain、Voiceflow、Dust などのツールを使って、ルーティングプロンプトを他の 2 つのプロンプトに接続します。

5. **Launch your solution（ソリューションの展開）**: チャットボットを Web サイトに展開する。ユーザーとボットのやり取りを継続的にモニタリングし、必要に応じてさらなる調整を行う。

## まとめ

GenAI Application Framework Methodology に従うことで、帽子に関するユーザーの問い合わせに効果的に対応するチャットボットを作成できました。

このプロセスは、ユーザーニーズの理解、ソリューションのテストと調整、ユーザーフィードバックに基づく継続的な改善の重要性を示しています。